In [1]:
import json
import statistics as stats
import pandas as pd
import puck.experiments.best_thresholding.hough_alt_plain as hap
import glob

In [58]:
output_file_dict = hap.main(
    input_path="../data/images_copy", 
    output_overall = "../output/experimental_results/", 
    output_results= "hough_options",
    output_bp ="hough_options_big_picture" ,
    output_timing ="../output/timing_results", 
    ground_truth="../data/annotations/annotations.json", 
    cli_printout = False
    )


100%|██████████| 48/48 [01:56<00:00,  2.42s/it]


In [35]:
output_file_dict

{'bigpicture0': '../output/experimental_results/hough_options_big_picture/houghCircle_results.csv',
 'bigpicture1': '../output/experimental_results/hough_options_big_picture/houghCircleAlt_results.csv',
 'results0': '../output/experimental_results/hough_options/houghCircle',
 'results1': '../output/experimental_results/hough_options/houghCircleAlt',
 'output_timing': '../output/timing_results/timingResults_hough_alt_plain_20260327_111702.txt'}

In [34]:
results0_folder_list = (glob.glob(f"{output_file_dict.get("results0")}/*"))
results1_folder_list = (glob.glob(f"{output_file_dict.get("results1")}/*"))

In [41]:
def big_picture_label_splitter(df):
    df.columns = ["parameters","choice","accuracy", "average_time_ns", "median_time_ns"]
    df[['dp','min_dist', 'p1', 'p2', 'min_radius']] = df.parameters.str.split(",",expand=True)
    df.min_radius =df.min_radius.str.strip(")")
    df.dp = df.dp.str[12:-1]
    df = df.drop(["choice",], axis=1)
    df = df.reindex(columns=["parameters","dp","min_dist", "p1", "p2", "min_radius","accuracy", "average_time_ns", "median_time_ns"])
    return df


In [42]:
hough_circle_csv = big_picture_label_splitter(pd.read_csv(output_file_dict.get("bigpicture0")))
hough_circle_alt_csv = big_picture_label_splitter(pd.read_csv(output_file_dict.get("bigpicture1")))

In [43]:
hough_circle_csv.sort_values("accuracy", ascending=False)

,parameters,dp,min_dist,p1,p2,min_radius,accuracy,average_time_ns,median_time_ns
28,"(np.float64(1.0), 250, 200, 30, 10)",1.0,250,200,30,10,0.156250,2.031098e+07,19233542.0
26,"(np.float64(1.0), 250, 150, 30, 10)",1.0,250,150,30,10,0.156250,1.995292e+07,19502146.0
60,"(np.float64(1.5), 250, 200, 30, 10)",1.5,250,200,30,10,0.156250,1.753964e+07,16896291.0
58,"(np.float64(1.5), 250, 150, 30, 10)",1.5,250,150,30,10,0.156250,2.133092e+07,20471896.0
20,"(np.float64(1.0), 200, 200, 30, 10)",1.0,200,200,30,10,0.154167,1.838018e+07,18227979.0
...,...,...,...,...,...,...,...,...,...
45,"(np.float64(1.5), 150, 200, 40, 10)",1.5,150,200,40,10,0.133333,2.096672e+07,20376292.0
11,"(np.float64(1.0), 150, 150, 40, 10)",1.0,150,150,40,10,0.129167,2.070405e+07,19924708.5
43,"(np.float64(1.5), 150, 150, 40, 10)",1.5,150,150,40,10,0.129167,2.153403e+07,20440062.5
3,"(np.float64(1.0), 100, 150, 40, 10)",1.0,100,150,40,10,0.129167,1.928888e+07,18704875.0


In [45]:
best_plain = hough_circle_csv.sort_values("accuracy", ascending=False).iloc[0]
best_alt = hough_circle_alt_csv.sort_values("accuracy", ascending=False).iloc[0]

In [54]:
best_plain_csv_file_name = best_plain["parameters"]+"hough_results.csv"
best_alt_csv_file_name = best_alt["parameters"]+"hough_results.csv"

best_plain_csv_file_name_full = [i for i in results0_folder_list if best_plain_csv_file_name in i][0]
best_alt_csv_file_name_full = [i for i in results1_folder_list if best_alt_csv_file_name in i][0]

In [22]:
def results_label_splitter(df):
    df.columns = ["name","dot_count","close_enough", "accuracy", "time_ns", "off_by"]
    df[["..","data","images", "palette", "loc", "height", "set", "file_name"]] = df.name.str.split("/",expand=True)
    df= df.drop(["..","data","images", "name"], axis=1)
    return df

In [55]:
best_plain_df = results_label_splitter(pd.read_csv(best_plain_csv_file_name_full))
best_plain_df 

,dot_count,close_enough,accuracy,time_ns,off_by,palette,loc,height,set,file_name
0,6,True,False,14186375,"[0.7071067811865476, 0.7071067811865476, 0.707...",custom,davids,high,A,custom_davids_high_A_0.jpg
1,7,True,False,13486250,"[0.7071067811865476, 0.7071067811865476, 1.581...",custom,davids,high,B,custom_davids_high_B_0.jpg
2,3,True,False,16661958,"[0.7071067811865476, 1.5811388300841898, 2.121...",custom,davids,high,C,custom_davids_high_C_0.jpg
3,4,True,True,19193709,"[0.7071067811865476, 0.7071067811865476, 1.581...",custom,davids,high,D,custom_davids_high_D_0.jpg
4,4,True,True,17234167,"[0.7071067811865476, 0.7071067811865476, 0.707...",custom,davids,medium,A,custom_davids_medium_A_0.jpg
...,...,...,...,...,...,...,...,...,...,...
91,4,True,True,19084208,"[0.7071067811865476, 1.5811388300841898, 2.121...",dark,michaels,medium,D,dark_michaels_medium_D_0.jpg
92,4,True,True,18552708,"[0.7071067811865476, 0.7071067811865476, 1.581...",dark,michaels,short,A,dark_michaels_short_A_0.jpg
93,4,True,True,19832334,"[0.7071067811865476, 2.5495097567963922, 2.915...",dark,michaels,short,B,dark_michaels_short_B_0.jpg
94,4,True,True,27521834,"[1.5811388300841898, 1.5811388300841898, 2.549...",dark,michaels,short,C,dark_michaels_short_C_0.jpg


In [56]:
best_alt_df = results_label_splitter(pd.read_csv(best_alt_csv_file_name_full))
best_alt_df

,dot_count,close_enough,accuracy,time_ns,off_by,palette,loc,height,set,file_name
0,5,True,False,129843000,"[0.0, 1.0, 1.0, 1.0, 885.353036929337]",custom,davids,high,A,custom_davids_high_A_0.jpg
1,5,True,False,132279083,"[1.0, 1.0, 2.23606797749979, 2.23606797749979,...",custom,davids,high,B,custom_davids_high_B_0.jpg
2,3,True,False,125388125,"[0.0, 1.0, 2.23606797749979]",custom,davids,high,C,custom_davids_high_C_0.jpg
3,4,True,True,146390167,"[1.4142135623730951, 1.4142135623730951, 1.414...",custom,davids,high,D,custom_davids_high_D_0.jpg
4,4,True,True,146388958,"[1.0, 1.0, 1.4142135623730951, 1.4142135623730...",custom,davids,medium,A,custom_davids_medium_A_0.jpg
...,...,...,...,...,...,...,...,...,...,...
91,4,True,True,124827042,"[1.4142135623730951, 2.0, 2.23606797749979, 2....",dark,michaels,medium,D,dark_michaels_medium_D_0.jpg
92,4,True,True,130786959,"[0.0, 1.0, 1.0, 4.242640687119285]",dark,michaels,short,A,dark_michaels_short_A_0.jpg
93,4,True,True,125187583,"[1.0, 2.0, 2.8284271247461903, 3.1622776601683...",dark,michaels,short,B,dark_michaels_short_B_0.jpg
94,4,True,True,139017208,"[1.0, 3.0, 3.0, 3.1622776601683795]",dark,michaels,short,C,dark_michaels_short_C_0.jpg
